# Instructions

Make a copy of this colab. Answer all questions by writing code. Run all cells and save output. Download .ipynb and submit it in canvas.

Please check back often for any updates.

2026/04/22 2:40pm: Initial version.

# Imports

In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import TensorDataset, DataLoader
import copy


In [11]:
if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  try:
    import torch_xla
    device = torch_xla.device()
  except ImportError:
    device = 'cpu'

device


'cuda'

In [12]:
try:
  import torchmetrics
except ImportError:
  !pip install -q torchmetrics
  import torchmetrics

In [13]:
def eval(model, loss_fn, metric, data_loader, transform=None):
  model.eval()
  metric.reset()
  total_loss = 0.0
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      if transform:
        X_batch = transform(X_batch)
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      total_loss += loss.item()
      metric.update(y_pred, y_batch)
  loss = total_loss / len(data_loader)
  return loss, metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, transform=None):
  history = {'train_losses': [], 'train_metrics': [],
             'valid_losses': [], 'valid_metrics': []}
  for epoch in range(n_epochs):
    total_loss = 0.0
    metric.reset()
    model.train()
    for X_batch, y_batch in train_loader:
      if transform:
        X_batch = transform(X_batch)
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)
    loss = total_loss / len(train_loader)
    history['train_losses'].append(loss)
    history['train_metrics'].append(metric.compute().item())
    valid_loss, valid_metric = eval(model, loss_fn, metric, valid_loader, transform)
    history['valid_losses'].append(valid_loss)
    history['valid_metrics'].append(valid_metric.item())

    print(f'Epoch {epoch + 1}/{n_epochs}, '
          f'train loss: {history['train_losses'][-1]:.2f}, '
          f'train metric: {history['train_metrics'][-1]:.2f}, '
          f'valid loss: {history['valid_losses'][-1]:.2f}, '
          f'valid metric: {history['valid_metrics'][-1]:.2f}')
  return history

# CIFAR-10 Data

In [26]:
# Get CIFAR-10 training data
train_data = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]))

# Get CIFAR-10 test data
test_data = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]))


In [19]:
# What are the shapes of train and test?
train_data.data.shape
test_data.data.shape



(10000, 32, 32, 3)

In [27]:
from torch.utils import data
# Create a dataloader for train and test.
batch_size = 64
data_loader_train = DataLoader(train_data, batch_size=batch_size, shuffle=True)
data_loader_test = DataLoader(test_data, batch_size=batch_size, shuffle=False)


# Build a simple CNN classifier

In [31]:
# Q1 (2 points) Build a simple CNN model1, check how many trainable params it
# has, train it for 3 epochs, measure accuracy.

model1 = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, 2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
)


In [23]:
model1

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=4096, out_features=512, bias=True)
  (8): ReLU()
  (9): Linear(in_features=512, out_features=10, bias=True)
)

In [32]:
trainable_params = sum(p.numel() for p in model1.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 2122186


In [29]:
model1 = model1.to(device)
optimizer1 = optim.Adam(model1.parameters(), lr=1e-3)
loss_fn1 = nn.CrossEntropyLoss()
metric1 = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
n_epochs1 = 3

history1 = train(model1, optimizer1, loss_fn1, metric1, data_loader_train, data_loader_test, n_epochs1)

Epoch 1/3, train loss: 1.42, train metric: 0.49, valid loss: 1.15, valid metric: 0.59
Epoch 2/3, train loss: 1.03, train metric: 0.63, valid loss: 1.00, valid metric: 0.64
Epoch 3/3, train loss: 0.86, train metric: 0.70, valid loss: 0.87, valid metric: 0.70


# Build classifier on top of ResNet50

In [36]:
# Get resnet50 model.

model2 = torchvision.models.resnet50(weights='IMAGENET1K_V1')


In [37]:
model2

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [38]:
# Check trainable params.

trainable_params = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 25557032


In [49]:
# Q2 (3 points) Freeze the model. Replace the top/last layer with a new layer
# with trainable params. Now how many trainable params and total params does
# it have? Train it for 1 epoch only (Remember to apply transform for ImageNet).
# Measure accuracy.


model3 = copy.deepcopy(model2)

# Freeze all parameters in the network
for param in model3.parameters():
    param.requires_grad = False

# Replace the top/last layer with a new layer
# ResNet50's classifier is `fc`
num_ftrs = model3.fc.in_features
model3.fc = nn.Linear(num_ftrs, 10) # 10 classes for CIFAR-10


In [50]:
model3

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [51]:
trainable_params = sum(p.numel() for p in model3.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 20490


In [52]:
total_params = sum(p.numel() for p in model3.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 23528522


In [53]:
transform_imagenet = T.Compose([
    T.Resize((224, 224)), T.ToImage(), T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])])

In [55]:
model3 = model3.to(device)
optimizer3 = optim.Adam(model3.parameters(), lr=1e-3)
loss_fn3 = nn.CrossEntropyLoss()
metric3 = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
n_epochs3 = 1
history3 = train(model3, optimizer3, loss_fn3, metric3, data_loader_train, data_loader_test, n_epochs3, transform_imagenet)

Epoch 1/1, train loss: 0.76, train metric: 0.75, valid loss: 0.59, valid metric: 0.80
